# Conformer-CTC demo

Enter a dataset URL and run the cells in Colab. A GPU is recommended.

Input: a ZIP/TAR archive or a Google Drive folder containing `audio/` and optional `transcriptions/`.

In [ ]:
DATASET_URL = ""  # @param {type:"string"}

## Install

For a private repository, add the GitHub token to Colab Secrets as `GITGUB_TOKEN` and grant this notebook access. For a public repository, set `PRIVATE = False`.

In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq build-essential cmake libsndfile1 gh aria2
python -m pip install -q uv==0.13.1 gdown==6.0.0

In [ ]:
from google.colab import userdata
import os
import subprocess

os.environ["MPLBACKEND"] = "Agg"
os.chdir("/content")

PRIVATE = False
GITHUB_URL = "https://github.com/Yaitco/DLA-1"
PYTHON = "/content/asr-venv/bin/python"

if PRIVATE:
    os.environ["GITHUB_TOKEN"] = userdata.get("GITGUB_TOKEN")
    subprocess.run(["gh", "repo", "clone", GITHUB_URL], check=True)
else:
    subprocess.run(["git", "clone", GITHUB_URL], check=True)

In [ ]:
%%bash
set -e
uv venv --python 3.11 --seed /content/asr-venv
/content/asr-venv/bin/python -m pip install -q -r /content/DLA-1/requirements-colab.txt

## Download resources

Download the checkpoint, tokenizer and sample from Hugging Face. Download the 4-gram LM from OpenSLR.

In [ ]:
%cd /content/DLA-1

import json
from pathlib import Path

from download_dataset import download_dataset

subprocess.run([PYTHON, "download_resources.py", "--lm-order", "4"], check=True)

## Sample inference

Run Conformer-CTC and print CER/WER, recording count and inference time (including model/LM loading). Metrics are displayed as percentages and saved as fractions in `scores.json` and `stats.json`.

In [ ]:
from time import perf_counter


def evaluate(data_dir, name):
    output = Path("data/saved/demo") / name
    predictions = output / "test"
    for path in predictions.glob("*.txt"):
        path.unlink()
    (output / "scores.json").unlink(missing_ok=True)

    started = perf_counter()
    subprocess.run(
        [
            PYTHON,
            "inference.py",
            "-cn",
            "inference_final_4gram",
            f"datasets.test.data_dir={json.dumps(str(data_dir))}",
            f"inferencer.save_path=demo/{name}",
        ],
        check=True,
    )
    elapsed = perf_counter() - started
    count = len(list(predictions.glob("*.txt")))
    stats = {
        "recordings": count,
        "inference_seconds": round(elapsed, 2),
        "recordings_per_second": count / elapsed,
    }

    references = data_dir / "transcriptions"
    if any(references.glob("*.txt")):
        subprocess.run(
            [
                PYTHON,
                "calc_metrics.py",
                "--ground-truth",
                str(references),
                "--predictions",
                str(predictions),
                "--output",
                str(output / "scores.json"),
            ],
            check=True,
            stdout=subprocess.DEVNULL,
        )
        stats.update(json.loads((output / "scores.json").read_text()))

    (output / "stats.json").write_text(json.dumps(stats, indent=2))
    print(
        f"{name}: {count} recordings, {elapsed:.1f} s, {count / elapsed:.2f} recordings/s"
    )
    for metric in ("CER", "WER", "corpus_CER", "corpus_WER"):
        if metric in stats:
            print(f"{metric}: {stats[metric]:.2%}")
    print(f"Results: {output.resolve()}")
    return output


evaluate(Path("resources/final/sample_dataset"), "sample")

## Dataset inference

Download your dataset and run the same evaluation. Predictions are saved in `data/saved/demo/custom/test/`. Without transcriptions, scoring is skipped.

In [ ]:
if DATASET_URL.strip():
    data_dir = download_dataset(DATASET_URL.strip(), workers=4)
    evaluate(data_dir, "custom")

## Download predictions

Download a ZIP with sample/custom predictions, metrics and inference configs.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/asr_predictions", "zip", "data/saved/demo")
files.download(archive)